In [1]:
import numpy as np
import faiss
import pandas as pd
import sys
sys.path.append('/Users/mac/Documents/Chat-Bot-Telegram/SQL')
import sql_query

In [2]:
#run venv and run it again

from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-07-29 23:42:56.718888: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
import pyodbc
import pandas as pd

def connect():
    return pyodbc.connect(
        "DRIVER={ODBC Driver 17 for SQL Server};"
        "SERVER=host.docker.internal,1433;"
        "DATABASE=ChatBot;"
        "UID=sa;"
        "PWD=Super@Password1234"
    )

def create(sender, message_text, vector):
    vector_str = ','.join(str(x) for x in vector)
    with connect() as conn:
        with conn.cursor() as cursor:
            query = "INSERT INTO Messages (Sender, MessageText, Vector) VALUES (?, ?, ?)"
            cursor.execute(query, (sender, message_text, vector_str))
            conn.commit()
            

def read_all():
    conn = connect()
    cursor = conn.cursor()
    query = "SELECT Id, Sender, MessageText, Vector FROM Messages"
    cursor.execute(query)
    rows = cursor.fetchall()
    result = []
    for row in rows:
        vector = [float(x) for x in row.Vector.split(',')] if row.Vector else None
        result.append({
            "Id": row.Id,
            "Sender": row.Sender,
            "MessageText": row.MessageText,
            "Vector": vector
        })
    cursor.close()
    conn.close()
    return result

def read_by_id(record_id):
    conn = connect()
    cursor = conn.cursor()
    query = "SELECT Id, Sender, MessageText, Vector FROM Messages WHERE Id = ?"
    cursor.execute(query, (record_id,))
    row = cursor.fetchone()
    if row:
        vector = [float(x) for x in row.Vector.split(',')] if row.Vector else None
        result = {
            "Id": row.Id,
            "Sender": row.Sender,
            "MessageText": row.MessageText,
            "Vector": vector
        }
    else:
        result = None
    cursor.close()
    conn.close()
    return result

def update(record_id, new_sender=None, new_message_text=None, new_vector=None):
    conn = connect()
    cursor = conn.cursor()

    fields = []
    params = []

    if new_sender is not None:
        fields.append("Sender = ?")
        params.append(new_sender)
    if new_message_text is not None:
        fields.append("MessageText = ?")
        params.append(new_message_text)
    if new_vector is not None:
        vector_str = ','.join(str(x) for x in new_vector)
        fields.append("Vector = ?")
        params.append(vector_str)

    if not fields:
        cursor.close()
        conn.close()
        return  

    params.append(record_id)
    query = f"UPDATE Messages SET {', '.join(fields)} WHERE Id = ?"
    cursor.execute(query, params)
    conn.commit()
    cursor.close()
    conn.close()

def delete(record_id):
    conn = connect()
    cursor = conn.cursor()
    query = "DELETE FROM Messages WHERE Id = ?"
    cursor.execute(query, (record_id,))
    conn.commit()
    cursor.close()
    conn.close()



chunk_size = 10000
nigga = True
checkchunk = 0
for chunk in pd.read_csv('/content/drive/MyDrive/chunks/all_information_only.csv', chunksize=chunk_size):
    vectors = model.encode(chunk['all_information'].reset_index(drop=True), batch_size=32, show_progress_bar=True)
    if nigga:
        np.save('/content/drive/MyDrive/chunks/vectors.npy', vectors)
        nigga = False
        checkchunk = 1
    else:
        last_vector = np.load('/content/drive/MyDrive/chunks/vectors.npy')
        all_vec = np.vstack([last_vector, vectors])
        np.save('/content/drive/MyDrive/chunks/vectors.npy', all_vec)
        checkchunk += 1
        print(checkchunk)
# for rowtext, vector in zip(chunk['all_information'], vectors):
#     create('user', rowtext, vector)




NameError: name 'model' is not defined

In [ ]:
prompt = input("لطفاً سوال خود را درباره محصول وارد کنید: ")

vector = model.encode(prompt)
